# Dự báo bụi mịn PM2.5 trước 24 giờ

**Sổ Nghiên Cứu AI, dự án mẫu 7/10.** So sánh mô hình lặp lại, hồi quy và LSTM với kiểm định theo thời gian.

**Câu hỏi nghiên cứu:** LSTM có dự báo PM2.5 trước 24 giờ tốt hơn mô hình “ngày mai giống hôm nay” không?

**Giả thuyết:** LSTM dùng thêm gió và độ ẩm cho MAE thấp hơn mô hình lặp lại ít nhất 20%.

### Cách dùng notebook này
1. Trên Colab, chọn **Runtime → Change runtime type → T4 GPU** nếu có (không bắt buộc).
2. Chạy lần lượt từng ô bằng **Shift + Enter**. Lần đầu có thể mất vài phút để cài thư viện và tải dữ liệu.
3. Các ô **Việc của em** là chỗ em tự làm tiếp để biến dự án mẫu thành nghiên cứu của riêng mình.

Dữ liệu tải từ Open-Meteo (miễn phí). Lưu ý: PM2.5 của Open-Meteo là **số liệu mô hình hóa** (CAMS), không phải trạm đo. Khi làm thật, nên đối chiếu với trạm quan trắc hoặc cảm biến em tự lắp.

> **Liêm chính khoa học.** Đây là mã mẫu để luyện tập. Nếu dùng cho dự án dự thi, hãy ghi rõ trong sổ nghiên cứu và báo cáo đoạn mã nào lấy từ notebook này, đoạn nào em tự viết hoặc sửa. Dữ liệu dự thi phải do em thu thập hoặc lấy từ nguồn công khai có ghi nguồn.

In [ ]:
import os, random, time, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
SEED = 42
random.seed(SEED); np.random.seed(SEED)
plt.rcParams["figure.dpi"] = 110
def tieu_de(s): print("\n" + "=" * 8, s, "=" * 8)

import requests, torch
from torch import nn
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error
_ = torch.manual_seed(SEED)

## 1. Dữ liệu theo giờ

In [ ]:
LAT, LON = 21.03, 105.85                                    # đổi thành nơi em nghiên cứu
het = (pd.Timestamp.today().normalize() - pd.Timedelta(days=8)).date(); dau = (pd.Timestamp(het) - pd.Timedelta(days=365)).date()
def tai():
    aq = requests.get(f"https://air-quality-api.open-meteo.com/v1/air-quality?latitude={LAT}&longitude={LON}&hourly=pm2_5&start_date={dau}&end_date={het}&timezone=auto", timeout=60).json()
    w = requests.get(f"https://archive-api.open-meteo.com/v1/archive?latitude={LAT}&longitude={LON}&start_date={dau}&end_date={het}&hourly=temperature_2m,relative_humidity_2m,wind_speed_10m,precipitation&timezone=auto", timeout=60).json()
    a = pd.DataFrame({"t": pd.to_datetime(aq["hourly"]["time"]), "pm25": aq["hourly"]["pm2_5"]})
    b = pd.DataFrame(w["hourly"]).rename(columns={"time": "t"}); b["t"] = pd.to_datetime(b["t"])
    return a.merge(b, on="t")
if os.path.exists("pm25.csv"):
    D = pd.read_csv("pm25.csv", parse_dates=["t"]); NGUON = "file của em"
else:
    try: D = tai(); NGUON = "Open-Meteo"
    except Exception as e:
        print("Không tải được (", type(e).__name__, "), dùng dữ liệu giả lập.")
        t = pd.date_range(dau, het, freq="h"); h = np.arange(len(t)); gio = t.hour.values; th = t.month.values
        gio_ = 8 + 4 * np.sin(2 * np.pi * h / 24 / 6) + np.random.normal(0, 2, len(t)); am = 70 + 15 * np.sin(2 * np.pi * h / 24) + np.random.normal(0, 5, len(t))
        mua_kho = np.isin(th, [11, 12, 1, 2, 3]); pm = np.zeros(len(t)); pm[0] = 30
        for i in range(1, len(t)): pm[i] = 0.92 * pm[i - 1] + 0.08 * (25 + 30 * mua_kho[i] + 10 * (gio[i] in (7, 8, 18, 19))) - 0.4 * (gio_[i] - 8) + np.random.normal(0, 3)
        D = pd.DataFrame({"t": t, "pm25": np.clip(pm, 2, None), "temperature_2m": 27 + 4 * np.sin(2 * np.pi * h / 24), "relative_humidity_2m": am, "wind_speed_10m": np.clip(gio_, 0, None), "precipitation": 0.0})
        NGUON = "GIẢ LẬP"
D = D.set_index("t").interpolate().dropna()
print(NGUON, "|", len(D), "giờ, từ", D.index.min().date(), "đến", D.index.max().date())
D.pm25.plot(figsize=(11, 3), lw=.6, title="PM2.5 (µg/m³)"); plt.show()

## 2. Bài toán: từ 48 giờ vừa qua, dự báo PM2.5 sau đúng 24 giờ
Chia theo thời gian: 70% đầu để học, 15% tiếp để kiểm định, 15% cuối để kiểm tra.

In [ ]:
NHIN, TRUOC = 48, 24
COT = ["pm25", "temperature_2m", "relative_humidity_2m", "wind_speed_10m", "precipitation"]
A = D[COT].values.astype(np.float32); n = len(A)
i_tr, i_va = int(n * .70), int(n * .85)
mu, sd = A[:i_tr].mean(0), A[:i_tr].std(0) + 1e-6; Z = (A - mu) / sd
idx = np.arange(NHIN, n - TRUOC)
y = A[idx + TRUOC - 1, 0]                                        # PM2.5 sau 24 giờ
tach = lambda a, b: idx[(idx + TRUOC - 1 >= a) & (idx + TRUOC - 1 < b)]
I_tr, I_va, I_te = tach(0, i_tr), tach(i_tr, i_va), tach(i_va, n)
ys = lambda I: A[I + TRUOC - 1, 0]
KQ = {}
def ghi(ten, I, p): KQ[ten] = {"MAE": mean_absolute_error(ys(I), p), "RMSE": mean_squared_error(ys(I), p) ** .5}
ghi("Lặp lại: bằng giờ hiện tại (đối chứng)", I_te, A[I_te - 1, 0])
ghi("Trung bình 24 giờ vừa qua", I_te, np.array([A[i - 24:i, 0].mean() for i in I_te]))
dac_trung = lambda I: np.hstack([Z[I - 1], Z[I - 24], Z[I - 48 + 1], Z[[np.arange(i - 24, i) for i in I]].mean(1),
                                 np.c_[np.sin(2 * np.pi * D.index[I + TRUOC - 1].hour / 24), np.cos(2 * np.pi * D.index[I + TRUOC - 1].hour / 24)]])
r = Ridge(alpha=1.0).fit(dac_trung(I_tr), ys(I_tr)); ghi("Hồi quy Ridge trên đặc trưng trễ", I_te, r.predict(dac_trung(I_te)))
print(pd.DataFrame(KQ).T.round(2))

## 3. LSTM

In [ ]:
EPOCHS = int(os.environ.get("SNC_EPOCHS", 15))
class M(nn.Module):
    def __init__(s): super().__init__(); s.l = nn.LSTM(len(COT), 48, batch_first=True); s.o = nn.Linear(48, 1)
    def forward(s, x): return s.o(s.l(x)[0][:, -1]).squeeze(-1)
cua_so = lambda I: torch.tensor(np.stack([Z[i - NHIN:i] for i in I]))
Xtr, Ytr = cua_so(I_tr), torch.tensor((ys(I_tr) - mu[0]) / sd[0]); Xva = cua_so(I_va)
m = M(); opt = torch.optim.Adam(m.parameters(), 2e-3); tot, best = None, 1e9
for ep in range(EPOCHS):
    m.train(); perm = torch.randperm(len(Xtr))
    for i in range(0, len(Xtr), 128):
        b = perm[i:i + 128]; opt.zero_grad(); nn.functional.mse_loss(m(Xtr[b]), Ytr[b]).backward(); opt.step()
    m.eval()
    with torch.no_grad(): mae = mean_absolute_error(ys(I_va), m(Xva).numpy() * sd[0] + mu[0])
    if mae < best: best, tot = mae, {k: v.clone() for k, v in m.state_dict().items()}       # dừng sớm theo tập kiểm định
    print(f"Vòng {ep+1}: MAE kiểm định {mae:.2f}")
m.load_state_dict(tot); m.eval()
with torch.no_grad(): p_lstm = m(cua_so(I_te)).numpy() * sd[0] + mu[0]
ghi("LSTM", I_te, p_lstm)
kq = pd.DataFrame(KQ).T.round(2); kq["Giảm MAE so với lặp lại"] = (1 - kq.MAE / kq.MAE.iloc[0]).map("{:.0%}".format); print(kq)

## 4. Đúng mức cảnh báo không?
Người dân quan tâm mức (tốt, trung bình, kém...) hơn con số. Ở đây dùng ngưỡng PM2.5 24 giờ theo thang AQI của Mỹ để minh họa; **Việt Nam có thang VN_AQI riêng**, em nên dùng thang của Việt Nam trong báo cáo.

In [ ]:
muc = lambda v: np.digitize(v, [12.0, 35.4, 55.4, 150.4])
print("Đúng mức cảnh báo: lặp lại %.1f%% | LSTM %.1f%%" % (100 * (muc(A[I_te - 1, 0]) == muc(ys(I_te))).mean(), 100 * (muc(p_lstm) == muc(ys(I_te))).mean()))
k = slice(0, 24 * 14)
plt.figure(figsize=(11, 3)); plt.plot(D.index[I_te + TRUOC - 1][k], ys(I_te)[k], label="thật", lw=1.2)
plt.plot(D.index[I_te + TRUOC - 1][k], A[I_te - 1, 0][k], label="lặp lại", lw=.8); plt.plot(D.index[I_te + TRUOC - 1][k], p_lstm[k], label="LSTM", lw=.8)
plt.legend(); plt.title("Hai tuần đầu của tập kiểm tra"); plt.show()

## Việc của em
1. Chạy LSTM 5 lần với `SEED` khác nhau, đưa MAE vào công cụ *Kiểm tra kết quả* để xem LSTM có hơn Ridge thật không.
2. Tự lắp cảm biến bụi (PMS5003 + ESP32), đo song song cạnh trạm chuẩn, rồi dùng hồi quy để hiệu chỉnh cảm biến. Đây là một hướng tạo tính mới tốt.
3. So sánh riêng mùa khô và mùa mưa.